# 🧠 TensorFlow & Keras: Zero to Hero — A Guided Lab

TensorFlow is Google's deep-learning framework; **Keras** is its high-level API that makes
building networks feel like stacking Lego. This lab takes you from tensors to a trained,
saved model.

**How this lab works**
- 📖 **Theory** → 🧠 **Mental model** → 🖼️ **ASCII diagram** → 🔬 **Worked example** →
  ⚡ **Pro tips** → ⚠️ **Traps** → ✏️ **Your Turn** → ✅ **Solution**.

**PyTorch vs TensorFlow (quick orientation).** They solve the same problems. PyTorch is
loved for research/flexibility; TensorFlow/Keras is loved for a clean high-level API and easy
deployment. If you know one, the concepts transfer directly.

**Install:** `pip install tensorflow` (or `tensorflow-cpu`)

**Roadmap**
1. Tensors & basic ops
2. Variables & GradientTape (autodiff)
3. The Keras Sequential API
4. Compile → Fit → Evaluate (the Keras workflow)
5. The Functional API (flexible models)
6. Real project: regression
7. Real project: classification
8. Callbacks (EarlyStopping, checkpoints)
9. Saving & loading models
10. 🏆 Capstone: end-to-end classifier


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # quiet TF logging
import tensorflow as tf
from tensorflow import keras
import numpy as np
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

---
## Chapter 1 — Tensors & Basic Ops

📖 **Theory.** A TensorFlow **tensor** is an n-dimensional array (like NumPy/PyTorch). TF
tensors are **immutable** — operations produce new tensors. You create them with
`tf.constant`, and TF ops (`tf.add`, `tf.matmul`, `*`, `@`) work elementwise or as matrix ops.

🧠 **Mental model.** Same "nested boxes" idea as NumPy: scalar → vector → matrix → higher-D.
TF adds automatic GPU/TPU execution and gradient tracking on top.

🖼️ **Diagram — tensor ranks**
```
 tf.constant(5)      tf.constant([1,2,3])    tf.constant([[1,2],[3,4]])
    rank 0                 rank 1                    rank 2
   shape ()              shape (3,)               shape (2, 2)
```


In [ ]:
a = tf.constant([[1., 2., 3.], [4., 5., 6.]])
print("tensor:\n", a.numpy())
print("shape:", a.shape, " dtype:", a.dtype)

b = tf.constant([[1., 0.], [0., 1.], [1., 1.]])
print("\nmatmul a@b:\n", (a @ b).numpy())
print("elementwise a*a:\n", (a*a).numpy())

# interop with NumPy is seamless
print("\nfrom numpy:", tf.constant(np.arange(5, dtype=np.float32)).numpy())
print("to numpy:", a.numpy().shape)

⚡ **Pro tip.** `.numpy()` converts any TF tensor back to a NumPy array — handy for
inspection and plotting.

⚠️ **Common trap.** TF is picky about dtypes: mixing `int32` and `float32` in an op raises an
error. Cast explicitly with `tf.cast(x, tf.float32)`.

### ✏️ Your Turn 1.1
1. Create a `(3, 3)` constant tensor of your choice.
2. Compute its transpose with `tf.transpose`.
3. Cast an integer tensor `tf.constant([1,2,3])` to float32.

In [ ]:
t = None
t_T = None
floated = None
print(t_T, floated)

✅ **Solution**
```python
t = tf.constant([[1.,2.,3.],[4.,5.,6.],[7.,8.,9.]])
t_T = tf.transpose(t)
floated = tf.cast(tf.constant([1,2,3]), tf.float32)
```

---
## Chapter 2 — Variables & GradientTape (Autodiff)

📖 **Theory.** Trainable parameters are **`tf.Variable`** objects (mutable tensors). To get
gradients, TF uses **`tf.GradientTape`**: any operation on a watched variable *inside the tape
context* is recorded, then `tape.gradient(loss, vars)` computes derivatives.

🧠 **Mental model.** `GradientTape` is a "recording session." You press record (enter the
`with` block), do your math, stop recording, then ask the tape for gradients.

🖼️ **Diagram — GradientTape**
```
 with tf.GradientTape() as tape:   ◄── start recording
     y = w * x + b                 ◄── ops on Variables are taped
     loss = (y - target)**2
 grads = tape.gradient(loss, [w,b]) ◄── replay backward -> gradients
```


In [ ]:
w = tf.Variable(2.0)
b = tf.Variable(1.0)
x = tf.constant(3.0)
target = tf.constant(10.0)

with tf.GradientTape() as tape:
    y = w * x + b            # 7
    loss = (y - target)**2   # 9

grads = tape.gradient(loss, [w, b])
print("loss:", loss.numpy())
print("dloss/dw:", grads[0].numpy())   # 2*(7-10)*3 = -18
print("dloss/db:", grads[1].numpy())   # 2*(7-10)   = -6

⚡ **Pro tip.** By default a tape is used **once** and only watches `tf.Variable`s. To watch a
constant tensor, call `tape.watch(x)`. For multiple gradient calls, use
`tf.GradientTape(persistent=True)`.

### ✏️ Your Turn 2.1
For `w = tf.Variable(4.0)` and `f = w**2 + 3*w + 1`, use a GradientTape to compute `df/dw`.
The analytic answer is `2w+3 = 11` at w=4.

In [ ]:
w = tf.Variable(4.0)
grad = None
print(grad)

✅ **Solution**
```python
w = tf.Variable(4.0)
with tf.GradientTape() as tape:
    f = w**2 + 3*w + 1
grad = tape.gradient(f, w)
print(grad.numpy())   # 11.0
```

---
## Chapter 3 — The Keras Sequential API

📖 **Theory.** **Keras** is TF's high-level API. The **Sequential** model is a linear stack of
layers — perfect for straightforward feed-forward networks. Common layers:
- `keras.layers.Dense(units, activation)` — a fully-connected layer
- activations: `"relu"` (hidden), `"softmax"` (multiclass output), `"sigmoid"` (binary)

🖼️ **Diagram — Sequential stack**
```
 Input(4) ─► Dense(16, relu) ─► Dense(8, relu) ─► Dense(3, softmax) ─► probs
```

🧠 **Mental model.** Sequential = a pipeline where data flows straight through, one layer
after another. (For branches/multiple inputs, you'll use the Functional API in Ch.5.)


In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(4,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(3, activation="softmax"),
])
model.summary()

⚡ **Pro tip.** `model.summary()` prints every layer, its output shape, and parameter count —
your first debugging tool for architecture mistakes.

### ✏️ Your Turn 3.1
Build a Sequential model for a **binary** classification problem with 10 input features:
`Dense(32, relu) → Dense(16, relu) → Dense(1, sigmoid)`. Print its summary.

In [ ]:
binary_model = None
# binary_model.summary()


✅ **Solution**
```python
binary_model = keras.Sequential([
    keras.layers.Input(shape=(10,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
binary_model.summary()
```

---
## Chapter 4 — Compile → Fit → Evaluate (the Keras workflow)

📖 **Theory.** Keras hides the training loop behind three verbs:
- **`compile(optimizer, loss, metrics)`** — configure how the model learns
- **`fit(X, y, epochs, batch_size, validation_split)`** — train
- **`evaluate` / `predict`** — test / run inference

🖼️ **Diagram — the Keras workflow**
```
 model.compile(opt, loss, metrics)   # set up
      │
 model.fit(X, y, epochs, ...)        # train (Keras runs the loop for you)
      │
 model.evaluate(Xtest, ytest)        # measure
 model.predict(Xnew)                 # use
```

🧠 **Mental model.** Where PyTorch makes you write the 5-step loop, Keras writes it for you —
`fit()` *is* that loop under the hood.


In [ ]:
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=800, n_features=4, n_classes=3, n_informative=3, random_state=0)
X = X.astype("float32")

# normalize
X = (X - X.mean(0)) / X.std(0)
Xtr, Xte, ytr, yte = X[:640], X[640:], y[:640], y[640:]

model = keras.Sequential([
    keras.layers.Input(shape=(4,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(3, activation="softmax"),
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",   # integer labels
              metrics=["accuracy"])

history = model.fit(Xtr, ytr, epochs=30, batch_size=32,
                    validation_split=0.2, verbose=0)
print("final train acc:", round(history.history["accuracy"][-1], 3))
print("final val acc:  ", round(history.history["val_accuracy"][-1], 3))

test_loss, test_acc = model.evaluate(Xte, yte, verbose=0)
print("test accuracy:  ", round(test_acc, 3))

⚠️ **Common trap — which loss?**
- Integer labels (0,1,2) → `"sparse_categorical_crossentropy"`
- One-hot labels ([0,1,0]) → `"categorical_crossentropy"`
- Binary (0/1) with sigmoid output → `"binary_crossentropy"`
Using the wrong one is a very common silent bug.

### ✏️ Your Turn 4.1
`fit()` returns a `history` object. Print the list of validation accuracy values across epochs
(`history.history["val_accuracy"]`) and find the best epoch (highest val accuracy).

In [ ]:
val_accs = None
best_epoch = None
print(best_epoch)

✅ **Solution**
```python
val_accs = history.history["val_accuracy"]
best_epoch = int(np.argmax(val_accs))
print("best epoch:", best_epoch, "val acc:", round(val_accs[best_epoch],3))
```

---
## Chapter 5 — The Functional API

📖 **Theory.** The **Functional API** builds models as a graph of layers, enabling things
Sequential can't: multiple inputs/outputs, shared layers, and branches. You call each layer
*like a function* on the previous tensor.

🖼️ **Diagram — functional wiring**
```
 inputs = Input(4)
    │
    ▼
 Dense(16, relu)(inputs)  ─► x
    │
    ▼
 Dense(3, softmax)(x)     ─► outputs
    │
 Model(inputs, outputs)
```


In [ ]:
inputs = keras.Input(shape=(4,))
x = keras.layers.Dense(16, activation="relu")(inputs)
x = keras.layers.Dense(8, activation="relu")(x)
outputs = keras.layers.Dense(3, activation="softmax")(x)

func_model = keras.Model(inputs=inputs, outputs=outputs)
func_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("Functional model built. Params:", func_model.count_params())

🧠 **Mental model.** Sequential is a subset of Functional. Reach for Functional the moment you
need anything non-linear in the *architecture* (skip connections, multi-input, etc.).

### ✏️ Your Turn 5.1
Rebuild the binary model from Ch.3 (10 inputs → 32 relu → 16 relu → 1 sigmoid) using the
**Functional API** instead of Sequential.

In [ ]:
# functional binary model
func_binary = None
print(func_binary.count_params() if func_binary is not None else None)

✅ **Solution**
```python
inp = keras.Input(shape=(10,))
h = keras.layers.Dense(32, activation="relu")(inp)
h = keras.layers.Dense(16, activation="relu")(h)
out = keras.layers.Dense(1, activation="sigmoid")(h)
func_binary = keras.Model(inp, out)
```

---
## Chapter 6 — Real Project: Regression

📖 **Theory.** For regression, the output layer is a single `Dense(1)` with **no activation**
(linear), and the loss is **`"mse"`** (mean squared error). Normalizing inputs is essential.


In [ ]:
np.random.seed(0)
n = 1000
size = np.random.uniform(500, 3500, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 50, n)
price = size*150 + bedrooms*8000 - age*300 + np.random.normal(0, 15000, n)

X = np.stack([size, bedrooms, age], axis=1).astype("float32")
y = price.astype("float32")
Xm, Xs = X.mean(0), X.std(0)
Xn = (X - Xm)/Xs
Xtr, Xte, ytr, yte = Xn[:800], Xn[800:], y[:800], y[800:]

reg = keras.Sequential([
    keras.layers.Input(shape=(3,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1)                      # linear output for regression
])
reg.compile(optimizer="adam", loss="mse", metrics=["mae"])
reg.fit(Xtr, ytr, epochs=40, batch_size=32, verbose=0)
mse, mae = reg.evaluate(Xte, yte, verbose=0)
print(f"test MAE: ${mae:,.0f}")

### ✏️ Your Turn 6.1
Predict the price of a **2000 sqft, 3-bed, 10-year-old** house. Normalize the input with
`Xm/Xs`, then call `reg.predict`.

In [ ]:
new_house = np.array([[2000, 3, 10]], dtype="float32")
predicted = None
print(predicted)

✅ **Solution**
```python
xn = (new_house - Xm)/Xs
predicted = reg.predict(xn, verbose=0)[0,0]
print(f"${predicted:,.0f}")
```

---
## Chapter 7 — Real Project: Classification (with a validation curve)

📖 **Theory.** We'll train a multiclass classifier and inspect the **learning curves** (train
vs validation accuracy over epochs) — the key diagnostic for over/underfitting.

🖼️ **Diagram — reading learning curves**
```
 acc
  │      ______ train
  │     /   ___ val        gap widening = OVERFITTING
  │    /  _/                curves both low/flat = UNDERFITTING
  │   /__/                  both high & close = GOOD
  └───────────────► epochs
```


In [ ]:
from sklearn.datasets import load_iris
data = load_iris()
Xi = data.data.astype("float32"); yi = data.target
# shuffle + normalize
rng = np.random.default_rng(0)
perm = rng.permutation(len(Xi)); Xi, yi = Xi[perm], yi[perm]
Xi = (Xi - Xi.mean(0))/Xi.std(0)
Xtr, Xte, ytr, yte = Xi[:120], Xi[120:], yi[:120], yi[120:]

clf = keras.Sequential([
    keras.layers.Input(shape=(4,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(3, activation="softmax"),
])
clf.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist = clf.fit(Xtr, ytr, epochs=60, validation_split=0.2, verbose=0)
print("final val acc:", round(hist.history["val_accuracy"][-1], 3))
print("test acc:", round(clf.evaluate(Xte, yte, verbose=0)[1], 3))

### ✏️ Your Turn 7.1
From `hist.history`, print the final training accuracy and final validation accuracy. Is the
gap small (good) or large (overfitting)?

In [ ]:
train_acc = None
val_acc = None
print(train_acc, val_acc)

✅ **Solution**
```python
train_acc = hist.history["accuracy"][-1]
val_acc = hist.history["val_accuracy"][-1]
# small gap -> good generalization
```

---
## Chapter 8 — Callbacks

📖 **Theory.** **Callbacks** hook into training to automate good practices:
- `EarlyStopping` — stop when validation stops improving (prevents overfitting + saves time)
- `ModelCheckpoint` — save the best model during training
- `ReduceLROnPlateau` — lower the learning rate when progress stalls

🧠 **Mental model.** Callbacks are event listeners: "when X happens during `fit`, do Y."


In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

clf2 = keras.Sequential([
    keras.layers.Input(shape=(4,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(3, activation="softmax"),
])
clf2.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = clf2.fit(Xtr, ytr, epochs=200, validation_split=0.2,
              callbacks=[early_stop], verbose=0)
print(f"stopped after {len(h2.history['loss'])} epochs (max was 200)")
print("test acc:", round(clf2.evaluate(Xte, yte, verbose=0)[1], 3))

⚡ **Pro tip.** `restore_best_weights=True` rolls back to the epoch with the best validation
score — so you keep the best model, not the last (possibly overfit) one.

### ✏️ Your Turn 8.1
Create an `EarlyStopping` callback that monitors `"val_accuracy"` (note: for accuracy, higher
is better, so set `mode="max"`) with `patience=8`.

In [ ]:
es = None
print(es)

✅ **Solution**
```python
es = keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max",
                                   patience=8, restore_best_weights=True)
```

---
## Chapter 9 — Saving & Loading Models

📖 **Theory.** Keras makes persistence trivial. The modern format is a single `.keras` file
containing architecture + weights + optimizer state:
- Save: `model.save("model.keras")`
- Load: `keras.models.load_model("model.keras")` — ready to predict immediately

(You can also save just weights with `model.save_weights(...)`.)


In [ ]:
clf2.save("/tmp/iris_keras.keras")
reloaded = keras.models.load_model("/tmp/iris_keras.keras")

orig = clf2.predict(Xte, verbose=0).argmax(1)
new  = reloaded.predict(Xte, verbose=0).argmax(1)
print("reloaded matches original:", bool((orig == new).all()))

⚠️ **Common trap.** Unlike PyTorch (where you rebuild the architecture then load weights),
`model.save(...)` in Keras stores *everything* — so `load_model` needs no architecture code.
Don't mix up the two frameworks' conventions.

### ✏️ Your Turn 9.1
Save the regression model `reg` from Ch.6 to `/tmp/house.keras`, reload it, and confirm it
predicts the same value for `Xte[:1]`.

In [ ]:
# save reg, reload, compare predictions on Xte[:1]


✅ **Solution**
```python
reg.save("/tmp/house.keras")
reg2 = keras.models.load_model("/tmp/house.keras")
print(np.allclose(reg2.predict(Xte[:1], verbose=0), reg.predict(Xte[:1], verbose=0)))
```

---
## 🏆 Chapter 10 — Capstone: End-to-End Classifier

Build, train (with EarlyStopping), evaluate, and save a classifier from scratch on the
breast-cancer dataset (binary classification, 30 features).

In [ ]:
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer()
Xb = bc.data.astype("float32"); yb = bc.target.astype("float32")
rng = np.random.default_rng(1)
perm = rng.permutation(len(Xb)); Xb, yb = Xb[perm], yb[perm]
Xb = (Xb - Xb.mean(0))/Xb.std(0)
split = 450
Xtr, Xte = Xb[:split], Xb[split:]
ytr, yte = yb[:split], yb[split:]
print("train/test:", Xtr.shape, Xte.shape, "| features:", Xb.shape[1])

### ✏️ Capstone Tasks
1. Build a Sequential model: `Dense(32, relu) → Dense(16, relu) → Dense(1, sigmoid)`.
2. Compile with `"adam"`, `"binary_crossentropy"`, metric `"accuracy"`.
3. Fit with `validation_split=0.2`, up to 200 epochs, using **EarlyStopping** (`patience=10`).
4. Report **test accuracy** (aim > 0.95).
5. Save to `/tmp/cancer.keras`.

In [ ]:
# Your full pipeline here


✅ **Capstone Solution**
```python
model = keras.Sequential([
    keras.layers.Input(shape=(30,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
model.fit(Xtr, ytr, epochs=200, validation_split=0.2, callbacks=[es], verbose=0)
test_acc = model.evaluate(Xte, yte, verbose=0)[1]
print(f"test accuracy: {test_acc:.3f}")
model.save("/tmp/cancer.keras")
```

🎉 **You're a TensorFlow/Keras practitioner!** You can build models two ways (Sequential &
Functional), run the compile→fit→evaluate workflow, use callbacks, and save/load models — and
you understand the low-level GradientTape underneath it all.

---
### 📌 Function Quick-Reference
**Tensors:** `tf.constant, tf.Variable, tf.matmul, tf.transpose, tf.cast, .numpy()`
**Autodiff:** `tf.GradientTape, tape.gradient, tape.watch`
**Build:** `keras.Sequential, keras.layers.Input/Dense, keras.Input, keras.Model`
**Train:** `model.compile(optimizer, loss, metrics), model.fit(epochs, batch_size, validation_split), model.summary()`
**Use:** `model.evaluate, model.predict, model.count_params`
**Callbacks:** `EarlyStopping, ModelCheckpoint, ReduceLROnPlateau`
**Persistence:** `model.save('x.keras'), keras.models.load_model`
**Losses:** `mse, binary_crossentropy, sparse_categorical_crossentropy, categorical_crossentropy`
